In [0]:
import uuid
from datetime import datetime, timedelta

import pandas as pd

In [0]:
class PurchaseOrderGenerator:

    def generate(
        self,
        inventory_df: pd.DataFrame,
        products_df: pd.DataFrame,
        suppliers_df: pd.DataFrame
    ) -> pd.DataFrame:

        purchase_orders = []

        product_supplier_map = (
            products_df[
                [
                    "product_id",
                    "supplier_id"
                ]
            ]
        )

        supplier_lead_time = (
            suppliers_df.set_index(
                "supplier_id"
            )["lead_time_days"]
            .to_dict()
        )

        inventory = inventory_df.merge(
            product_supplier_map,
            on="product_id",
            how="inner"  # Changed from 'left' to 'inner' to only keep products with supplier assignments
        )
        
        # Filter out any rows with NaN supplier_id as a safety measure
        inventory = inventory.dropna(subset=['supplier_id'])

        for _, row in inventory.iterrows():

            avg_daily_sales = (
                row["inventory_qty"]
                / max(
                    row["days_of_cover"],
                    1
                )
            )

            lead_time = supplier_lead_time[
                row["supplier_id"]
            ]

            # TODO Phase 5: Replace simple reorder threshold with safety stock formula
            # Reorder Point = Lead Time Demand + Safety Stock
            # Where: Safety Stock = Z × σ × √LeadTime
            # 
            # Current logic (sufficient for B3 MVP):
            # Reorder point = lead time demand + 25-day safety stock
            # With inventory at 35-40 days and lead times at 3-14 days:
            #   - Reorder point ranges from 28-39 days
            #   - PO triggers when: days_of_cover < lead_time + 25
            #   - Expected PO rate: ~30-50% (positions with longer lead times)
            safety_stock_days = 25
            reorder_point = (
                avg_daily_sales * (lead_time + safety_stock_days)
            )

            if row["inventory_qty"] < reorder_point:

                order_qty = int(
                    reorder_point * 3
                )

                po_date = datetime.now()

                purchase_orders.append({

                    "po_number":
                        f"PO-{uuid.uuid4().hex[:10].upper()}",

                    "supplier_id":
                        row["supplier_id"],

                    "product_id":
                        row["product_id"],

                    "ordered_qty":
                        order_qty,

                    "received_qty":
                        None,

                    "lead_time_days":
                        lead_time,

                    "order_date":
                        po_date.date(),

                    "expected_delivery_date":
                        (
                            po_date +
                            timedelta(
                                days=lead_time
                            )
                        ).date(),

                    "actual_delivery_date":
                        None,

                    "po_status":
                        "CREATED",

                    "created_at":
                        datetime.now(),

                    "updated_at":
                        datetime.now()
                })

        return pd.DataFrame(
            purchase_orders
        )